### 1193. Monthly Transactions I
```
Table: Transactions

+---------------+---------+
| Column Name   | Type    |
+---------------+---------+
| id            | int     |
| country       | varchar |
| state         | enum    |
| amount        | int     |
| trans_date    | date    |
+---------------+---------+
id is the primary key of this table.
The table has information about incoming transactions.
The state column is an enum of type ["approved", "declined"].
 

Write an SQL query to find for each month and country, the number of transactions and their total amount, the number of approved transactions and their total amount.

Return the result table in any order.

The query result format is in the following example.

 

Example 1:

Input: 
Transactions table:
+------+---------+----------+--------+------------+
| id   | country | state    | amount | trans_date |
+------+---------+----------+--------+------------+
| 121  | US      | approved | 1000   | 2018-12-18 |
| 122  | US      | declined | 2000   | 2018-12-19 |
| 123  | US      | approved | 2000   | 2019-01-01 |
| 124  | DE      | approved | 2000   | 2019-01-07 |
+------+---------+----------+--------+------------+
Output: 
+----------+---------+-------------+----------------+--------------------+-----------------------+
| month    | country | trans_count | approved_count | trans_total_amount | approved_total_amount |
+----------+---------+-------------+----------------+--------------------+-----------------------+
| 2018-12  | US      | 2           | 1              | 3000               | 1000                  |
| 2019-01  | US      | 1           | 1              | 2000               | 2000                  |
| 2019-01  | DE      | 1           | 1              | 2000               | 2000                  |
+----------+---------+-------------+----------------+--------------------+-----------------------+

```

In [14]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType,StructField,IntegerType,StringType
from pyspark.sql.functions import col,date_format,when,lit,count,sum

In [3]:
spark=SparkSession.builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [5]:
Data = [[121, 'US', 'approved', 1000, '2018-12-18'], [122, 'US', 'declined', 2000, '2018-12-19'],
        [123, 'US', 'approved', 2000, '2019-01-01'], [124, 'DE', 'approved', 2000, '2019-01-07']]
Schema=StructType([StructField("id",IntegerType(),True),\
                  StructField("country",StringType(),True),\
                   StructField("state",StringType(),True),\
                  StructField("amount",IntegerType(),True),\
                  StructField("trans_date",StringType(),True)]
                  
                 )
Transactions=spark.createDataFrame(Data,Schema)
Transactions=Transactions.withColumn("trans_date",col("trans_date").cast("date"))


In [6]:
Transactions.show()

+---+-------+--------+------+----------+
| id|country|   state|amount|trans_date|
+---+-------+--------+------+----------+
|121|     US|approved|  1000|2018-12-18|
|122|     US|declined|  2000|2018-12-19|
|123|     US|approved|  2000|2019-01-01|
|124|     DE|approved|  2000|2019-01-07|
+---+-------+--------+------+----------+



In [18]:
Transactions=Transactions.withColumn("month",date_format("trans_date","yyyy-MM"))
Transactions.show()

+---+-------+--------+------+----------+-------+
| id|country|   state|amount|trans_date|  month|
+---+-------+--------+------+----------+-------+
|121|     US|approved|  1000|2018-12-18|2018-12|
|122|     US|declined|  2000|2018-12-19|2018-12|
|123|     US|approved|  2000|2019-01-01|2019-01|
|124|     DE|approved|  2000|2019-01-07|2019-01|
+---+-------+--------+------+----------+-------+



In [25]:
Result=(Transactions.groupBy("country","month")
    .agg(
        count(col('id')).alias("trans_count")
              ,sum(when(col("state")=="approved",col("amount")).otherwise(0)).alias("approved_total_amount")
               ,sum(col("amount")).alias("trans_total_amount")
              ,sum(when(col("state")=="approved",1).otherwise(0)).alias("approved_count")
       ).
    select(
        col("month"),
        col("country"),
        col("trans_count"),
        col("approved_count"),
        col("trans_total_amount"),
        col("approved_total_amount")
    )
       )

In [26]:
Result.show()

+-------+-------+-----------+--------------+------------------+---------------------+
|  month|country|trans_count|approved_count|trans_total_amount|approved_total_amount|
+-------+-------+-----------+--------------+------------------+---------------------+
|2018-12|     US|          2|             1|              3000|                 1000|
|2019-01|     US|          1|             1|              2000|                 2000|
|2019-01|     DE|          1|             1|              2000|                 2000|
+-------+-------+-----------+--------------+------------------+---------------------+



In [27]:
spark.stop()